In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:20px;}
</style>
"""))

In [11]:
import pandas as pd
from pathlib import Path

In [12]:
DATA_DIR = Path(r"C:\ai\source\09_project\data")
file_path = DATA_DIR / "공공자전거 대여소별 이용정보_2020.07~2021.01.csv"
print("파일유무:",file_path.exists())

파일유무: True


In [13]:
def read_csv_auto(path):
    for enc in ["utf-8-sig", "cp949"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"인코딩을 읽을 수 없음: {path.name}")

raw = read_csv_auto(file_path)

In [14]:
rename_map = {"대여소 그룹": "자치구", "일자 / 월": "대여년월",
              "대여 일자 / 월": "대여년월", "대여 건수": "대여건수"}
df = raw.copy()
df.columns = df.columns.str.strip()
df = df.rename(columns=rename_map)[["자치구", "대여소 명", "대여년월", "대여건수"]]

In [15]:
df["대여년월"] = df["대여년월"].astype(str).str.strip()

In [16]:
jan21 = df[df["대여년월"] == "202101"].copy()

In [17]:
jan21 = jan21[~jan21["자치구"].isin(["그룹명 없음", "정비센터"])]

In [18]:
jan = df[df["대여 일자 / 월"] == 202101].copy()
print(jan.shape)       



KeyError: '대여 일자 / 월'

In [19]:
from pathlib import Path
import pandas as pd

In [22]:
DATA_DIR = Path(r"C:\ai\source\09_project\data")
src = DATA_DIR / "공공자전거 대여소별 이용정보_2020.07~2021.01.csv"
dst = DATA_DIR / "공공자전거_대여소별_이용정보_202101.csv"


In [23]:
df = pd.read_csv(src, encoding="cp949")


In [24]:
jan = df[df["대여 일자 / 월"] == 202101].copy()
print(jan.shape)

(2190, 4)


In [25]:
jan.to_csv(dst, index=False, encoding="utf-8-sig")
print("저장 완료:", dst)

저장 완료: C:\ai\source\09_project\data\공공자전거_대여소별_이용정보_202101.csv


In [26]:
from pathlib import Path
import pandas as pd


In [27]:
DATA_DIR = Path(r"C:\ai\source\09_project\data")
files = [
    "공공자전거_대여소별_이용정보_202101.csv",        # utf-8-sig
    "공공자전거 대여소별 이용정보_21.02~21.06.csv",   # cp949, 끝에 빈 행 다수
    "대여소별 이용정보(월별 '21.7-'21.12).csv",       # cp949, 위에 안내 행 3줄
]
for f in files:
    print(f, "존재:", (DATA_DIR / f).exists())


공공자전거_대여소별_이용정보_202101.csv 존재: True
공공자전거 대여소별 이용정보_21.02~21.06.csv 존재: True
대여소별 이용정보(월별 '21.7-'21.12).csv 존재: True


In [36]:
def read_bike_csv(path):
    
    enc = None
    for e in ["utf-8-sig", "cp949"]:
        try:
            preview = pd.read_csv(path, encoding=e, header=None, nrows=20)
            enc = e
            break
        except UnicodeDecodeError:
            continue
    if enc is None:
        raise ValueError(f"인코딩을 읽을 수 없음: {path.name}")

    
    header_row = None
    for i, row in preview.iterrows():
        vals = row.astype(str).str.strip().tolist()
        if "대여소 그룹" in vals and "대여 건수" in vals:
            header_row = i
            break
    if header_row is None:
        raise ValueError(f"헤더를 찾을 수 없음: {path.name}")

    
    df = pd.read_csv(path, encoding=enc, header=header_row)
    print(f"{path.name}: 인코딩={enc}, 헤더={header_row}번 줄, 크기={df.shape}")
    return df


In [41]:
for f in files:
    df = read_bike_csv(DATA_DIR / f)
    print(f, [repr(c) for c in df.columns])

print(rename_map)

공공자전거_대여소별_이용정보_202101.csv: 인코딩=utf-8-sig, 헤더=0번 줄, 크기=(2190, 4)
공공자전거_대여소별_이용정보_202101.csv ["'대여소 그룹'", "'대여소 명'", "'대여 일자 / 월'", "'대여 건수'"]
공공자전거 대여소별 이용정보_21.02~21.06.csv: 인코딩=cp949, 헤더=0번 줄, 크기=(14749, 4)
공공자전거 대여소별 이용정보_21.02~21.06.csv ["'대여소 그룹'", "'대여소 명'", "'대여 일자 / 월'", "'대여 건수'"]
대여소별 이용정보(월별 '21.7-'21.12).csv: 인코딩=cp949, 헤더=3번 줄, 크기=(15148, 4)
대여소별 이용정보(월별 '21.7-'21.12).csv ["'대여소 그룹'", "'대여소 명'", "'대여 일자 / 월'", "'대여 건수'"]
{'대여소 그룹': '자치구', '대여 건수': '대여건수', '대여 일자/월': '대여년월'}


In [44]:
def standardize_columns(df):
    new_cols = {}
    for c in df.columns:
        key = "".join(str(c).split())
        if "그룹" in key:
            new_cols[c] = "자치구"
        elif "대여소명" in key:
            new_cols[c] = "대여소 명"
        elif "일자" in key or "년월" in key:
            new_cols[c] = "대여년월"
        elif "대여건수" in key:
            new_cols[c] = "대여건수"
    return df.rename(columns=new_cols)

use_cols = ["자치구", "대여소 명", "대여년월", "대여건수"]

frames = []
for f in files:
    df = read_bike_csv(DATA_DIR / f)
    print(f, "원래 컬럼:", [repr(c) for c in df.columns])
    df = standardize_columns(df)
    missing = [c for c in use_cols if c not in df.columns]
    if missing:
        print("⚠ 없는 컬럼:", missing)
        continue
    frames.append(df[use_cols])
    print("   → OK", df.shape)

공공자전거_대여소별_이용정보_202101.csv: 인코딩=utf-8-sig, 헤더=0번 줄, 크기=(2190, 4)
공공자전거_대여소별_이용정보_202101.csv 원래 컬럼: ["'대여소 그룹'", "'대여소 명'", "'대여 일자 / 월'", "'대여 건수'"]
   → OK (2190, 4)
공공자전거 대여소별 이용정보_21.02~21.06.csv: 인코딩=cp949, 헤더=0번 줄, 크기=(14749, 4)
공공자전거 대여소별 이용정보_21.02~21.06.csv 원래 컬럼: ["'대여소 그룹'", "'대여소 명'", "'대여 일자 / 월'", "'대여 건수'"]
   → OK (14749, 4)
대여소별 이용정보(월별 '21.7-'21.12).csv: 인코딩=cp949, 헤더=3번 줄, 크기=(15148, 4)
대여소별 이용정보(월별 '21.7-'21.12).csv 원래 컬럼: ["'대여소 그룹'", "'대여소 명'", "'대여 일자 / 월'", "'대여 건수'"]
   → OK (15148, 4)


In [46]:
rename_map = {
    "대여소 그룹":"자치구",
    "대여 건수":"대여건수",
    "대여 일자 / 월":"대여년월",
    "대여 건수":"대여건수"
}
use_cols = ["자치구","대여소 명","대여년월","대여건수"]
frames = []
for f in files:
    df = read_bike_csv(DATA_DIR / f)
    df.columns = df.columns.astype(str).str.strip()
    df = df.rename(columns=rename_map)[use_cols]
    frames.append(df)


공공자전거_대여소별_이용정보_202101.csv: 인코딩=utf-8-sig, 헤더=0번 줄, 크기=(2190, 4)
공공자전거 대여소별 이용정보_21.02~21.06.csv: 인코딩=cp949, 헤더=0번 줄, 크기=(14749, 4)
대여소별 이용정보(월별 '21.7-'21.12).csv: 인코딩=cp949, 헤더=3번 줄, 크기=(15148, 4)


In [47]:
bike21 = pd.concat(frames, ignore_index=True)
print("병합 직후:", bike21.shape)


병합 직후: (32087, 4)


In [48]:
bike21 = bike21.dropna(subset=use_cols)
print("빈 행 제거 후:", bike21.shape)



빈 행 제거 후: (29114, 4)


In [49]:
bike21["자치구"] = bike21["자치구"].astype(str).str.strip()
bike21["대여소 명"] = bike21["대여소 명"].astype(str).str.strip()
bike21["대여년월"] = pd.to_numeric(bike21["대여년월"], errors="coerce").astype("Int64").astype(str)
bike21["대여건수"] = pd.to_numeric(
    bike21["대여건수"].astype(str).str.replace(",", ""), errors="coerce"
)


In [50]:
before = len(bike21)
bike21 = bike21[~bike21["자치구"].isin(["그룹명 없음", "정비센터"])]
bike21 = bike21[~bike21["대여소 명"].isin(["", "대여소명 없음", "nan"])]
bike21 = bike21.dropna(subset=["대여건수"])
print("제거된 행:", before - len(bike21))



제거된 행: 119


In [51]:
keys = ["자치구", "대여소 명", "대여년월"]
dup = bike21.duplicated(subset=keys).sum()
print("중복 행 수:", dup)
if dup:
    bike21 = bike21.groupby(keys, as_index=False)["대여건수"].sum()



중복 행 수: 0


In [52]:
bike21["대여건수"] = bike21["대여건수"].astype(int)
bike21 = bike21.sort_values(["대여년월", "자치구", "대여소 명"]).reset_index(drop=True)

bike21.info()
print(bike21.head())
print(bike21["대여년월"].value_counts().sort_index())
print("자치구 수:", bike21["자치구"].nunique())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28995 entries, 0 to 28994
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   자치구     28995 non-null  object
 1   대여소 명   28995 non-null  object
 2   대여년월    28995 non-null  object
 3   대여건수    28995 non-null  int32 
dtypes: int32(1), object(3)
memory usage: 793.0+ KB
   자치구                           대여소 명    대여년월  대여건수
0  강남구                2301. 현대고등학교 건너편  202101   711
1  강남구  2302. 교보타워 버스정류장(신논현역 3번출구 후면)  202101   375
2  강남구                  2303. 논현역 7번출구  202101   309
3  강남구         2304. 신영 ROYAL PALACE 앞  202101   113
4  강남구              2305. MCM 본사 직영점 앞  202101    99
202101    2183
202102    2227
202103    2290
202104    2362
202105    2392
202106    2461
202107    2464
202108    2483
202109    2495
202110    2526
202111    2536
202112    2576
Name: 대여년월, dtype: int64
자치구 수: 25


In [53]:
out_path = DATA_DIR / "공공자전거_대여소별_이용정보_2021_전처리.csv"
bike21.to_csv(out_path, index=False, encoding="utf-8-sig")
print("저장 완료:", out_path)


저장 완료: C:\ai\source\09_project\data\공공자전거_대여소별_이용정보_2021_전처리.csv
